In [48]:
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict
from typing import Annotated, Literal, Union
from langgraph.types import CachePolicy, Send, Command
from langgraph.cache.memory import InMemoryCache
from datetime import datetime
import operator

In [23]:
# 이 정의에 없는 key는 무시가 된다. 그렇기에 잘 정의해 줘야 됨
class State(TypedDict):
    hello: str
    a: bool

graph_builder = StateGraph(State)

In [ ]:
# 함수의 return 값이 state를 업데이트 해준다.
def node_one(state: State):
    print("node_one:", state)
    return {"hello": "from node one.", "a": True}

def node_two(state: State):
    print("node_two:", state)
    return {"hello": "from node two."}

def node_three(state: State):
    print("node_three:", state)
    return {"hello": "from node three."}

# node_one이라는 이름으로 node_one 함수를 연결
graph_builder.add_node("node_one", node_one)
graph_builder.add_node("node_two", node_two)
graph_builder.add_node("node_three", node_three)

# START와 END는 예약어이며, 아래처럼 입력하면 순서대로 from to 순서로 연결이 된다
graph_builder.add_edge(START, "node_one")
graph_builder.add_edge("node_one", "node_two")
graph_builder.add_edge("node_two", "node_three")
graph_builder.add_edge("node_three", END)

# compile하면 그래프가 정상적으로 연결되었는지 검사
graph = graph_builder.compile()
# 그래프를 그림으로 보여줌 (이 방식은 langgraph 서버에서 그려 가져오는 방식)
graph
# 서버 호출 없이 그래프를 그리고 싶다면 아래 방식 사용
# print(graph.get_graph().draw_ascii())
# graph안에 포함되어 있는 함수를 실행
graph.invoke({'hello': 'world'})

In [29]:
# node 내부에서만 사용하는 state로 만들기
# 이렇게 하는 이유는 유저가 굳이 확인하지 않아도 되는 데이터를 따로 관리하기 위해
class PrivateState(TypedDict):
    a: int
    b: int
    c: int

# input state (User의 입력 값 정의)
class InputState(TypedDict):
    hello: str

# ouput state (최종 output 정의)
class OutputState(TypedDict):
    bye: str

class MegaPrivate(TypedDict):
    secret: bool

graph_builder = StateGraph(
    PrivateState,
    input_schema=InputState,
    output_schema=OutputState
)

In [ ]:
def node_one(state: InputState) -> InputState:
    print("node_one ->", state)
    return {"hello": "world"}

def node_two(state: PrivateState) -> PrivateState:
    print("node_two ->", state)
    return {"a": 1}

def node_three(state: PrivateState) -> PrivateState:
    print("node_three ->", state)
    return {"b": 1}

def node_four(state: PrivateState) -> OutputState:
    print("node_four ->", state)
    return {"bye": "world"}

def node_five(stat: OutputState):
    return {"secret": True}

def node_six(state: MegaPrivate):
    print(state)

graph_builder.add_node("node_one", node_one)
graph_builder.add_node("node_two", node_two)
graph_builder.add_node("node_three", node_three)
graph_builder.add_node("node_four", node_four)
graph_builder.add_node("node_five", node_five)
graph_builder.add_node("node_six", node_six)

graph_builder.add_edge(START, "node_one")
graph_builder.add_edge("node_one", "node_two")
graph_builder.add_edge("node_two", "node_three")
graph_builder.add_edge("node_three", "node_four")
graph_builder.add_edge("node_four", "node_five")
graph_builder.add_edge("node_five", "node_six")
graph_builder.add_edge("node_six", END)

graph = graph_builder.compile()
# graph
graph.invoke({'hello': 'world'})

In [ ]:
def update_function(old, new):
    return old + new

class State(TypedDict):
    # Annotated를 사용해서 state를 업데이트할 수 있음
    # state를 업데이트 하는 방법은 2번째 파라미터로 주어진 함수를 통해 할 수 있음, 해당 함수는 old, new 2개의 파라미터를 받을 수 있어야됨
    messages: Annotated[list[str], update_function]

graph_builder = StateGraph(State)

In [ ]:
def node_one(state: State):
    last_message = state["messages"][-1]
    # 첫번째 그래프에서 state에 있는 필드를 업데이트 하면 완전 덮어쓰기가 됨
    return {"messages": ["Hello! nice to meet you"]}

def node_two(state: State):
    return {}

def node_three(state: State):
    print("node_three ->", state)
    return {}

graph_builder.add_node("node_one", node_one)
# ttl로 설정한 시간동안 캐시가 유지되며 해당 시간 이전에 이 노드가 실행되면 캐시에 저장된 데이터를 줌
graph_builder.add_node("node_two", node_two, cache_policy=CachePolicy(ttl=20))
graph_builder.add_node("node_three", node_three)

graph_builder.add_edge(START, "node_one")
graph_builder.add_edge("node_one", "node_two")
graph_builder.add_edge("node_two", "node_three")
graph_builder.add_edge("node_three", END)

graph = graph_builder.compile(cache=InMemoryCache)

graph.invoke({"messages": ["Hello!"]})

In [44]:
# 13.7
class State(TypedDict):
    seed: int

graph_builder = StateGraph(State)

In [ ]:
def node_one(state: State):
    print("node_one ->", state)
    return {}

def node_two(state: State):
    print("node_two ->", state)
    return {}

def node_three(state: State):
    print("node_three ->", state)
    return {}

def node_four(state: State):
    print("node_four ->", state)
    return {}

graph_builder.add_node("node_one", node_one)
graph_builder.add_node("node_two", node_two)
graph_builder.add_node("node_three", node_three)
graph_builder.add_node("node_four", node_four)

# def decide_path(state: State) -> Literal["node_three", "node_four"]:
#     if state["seed"] % 2 == 0:
#         return "node_three"
#     else:
#         return "node_four"

def decide_path(state: State):
    return state["seed"] % 2 == 0

graph_builder.add_conditional_edges(
    START,
    decide_path,
    {True: "node_one", False: "node_two", "Hello": END}
)
graph_builder.add_edge("node_one", "node_two")
# 함수의 결과에 따라 어느 노드로 갈지는 3번째 파라미터의 객체 값으로 결정!
graph_builder.add_conditional_edges(
    "node_two", 
    decide_path, 
    {True: "node_three", False: "node_four", "Hello": END}
)
graph_builder.add_edge("node_four", END)

graph = graph_builder.compile()

graph.invoke({"seed": 2})

{'seed': 2}

In [ ]:
# node를 몇번 실행해야 될 지 모를 때 send를 사용해서 자동으로 실행할 수 있게 만들 수 있다.
class State(TypedDict):
    words: list[str]
    output: Annotated[list[dict[str, Union[str, int]]], operator.add]

graph_builder = StateGraph(State)

def node_one(state: State):
    print(f"I want to count {len(state["words"])} words in my state.")
    return {}

def node_two(word: str):
    return {
        "output": [{ "word": word, "letters": len(word) }]
    }

# def node_three(state: State):
#     print("node_three ->", state)
#     return {}

# def node_four(state: State):
#     print("node_four ->", state)
#     return {}

graph_builder.add_node("node_one", node_one)
graph_builder.add_node("node_two", node_two)
# graph_builder.add_node("node_three", node_three)
# graph_builder.add_node("node_four", node_four)

def dispatcher(state: State):
    # send는 node_two를 word의 수만큼 실행함
    return [Send("node_two", word) for word in state["words"]]

graph_builder.add_edge(START, "node_one")
graph_builder.add_conditional_edges(
    "node_one",
    dispatcher,
    ["node_two"]
)
# graph_builder.add_edge("node_two", "node_three")
graph_builder.add_edge("node_two", END)

graph = graph_builder.compile()

graph.invoke({
    "words": ["hello", "world", "how", "are", "you", "doing"]
})

In [ ]:
# command 다른 node로 전달할 때 사용할 수 있다
class State(TypedDict):
    transter_reason: str

graph_builder = StateGraph(State)

def triage_node(state: State) -> Command[Literal["account_support", "tech_support"]]:
    # command는 state를 업데이트 하면서 노드를 이동할 때 사용한다.
    return Command(goto="account_support", update={"transter_reason": "The user wants to change password."})

def tech_support(state: State):
    return {}

def account_support(state: State):
    return {}

graph_builder.add_node("triage_node", triage_node)
graph_builder.add_node("tech_support", tech_support)
graph_builder.add_node("account_support", account_support)

graph_builder.add_edge(START, "triage_node")
graph_builder.add_edge("tech_support", END)
graph_builder.add_edge("account_support", END)

graph = graph_builder.compile()
graph.invoke({})

{'transter_reason': 'The user wants to change password.'}